In [0]:
print("")   
%pip install nltk

In [0]:
### VECTORIZATION **TF-IDF** with library `scikit-learn`

### Código de Vectorización TF-IDF


from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import pandas as pd
from pathlib import Path
import string
import nltk
import re




# 1. Load the Dataset 
from pathlib import Path
DATA_DIR = Path("../data") #./data Actual level ---- ../data 2 levels up
# DATA_DIR = Path("/dbfs/workspace/datasets/imdb_big_dataset")
movies_path = DATA_DIR / "movies_final.csv"


# 0. Loading datasets imbd and links 
# Those two datasets are the conection between each movie and its information
movies_df = pd.read_csv(movies_path)
# movies_final = pd.read_csv("/dbfs/mnt/tu_ruta/data/movies_clean.csv")

# 1. Clean stopwords
stopwords = nltk.corpus.stopwords.words('english')
def clean_text(text):
    text = "".join([word for word in text if word not in string.punctuation])
    tokens = re.split('\\W+', text)
    text = [word for word in tokens if word not in stopwords]
    return text

#Apply the function
movies_df['combined_features_nostop'] = movies_df['combined_features'].apply(lambda x: clean_text(x.lower()))
movies_df.head(5)

In [0]:
print(len(movies_df.columns))
movies_df.columns

In [0]:
ps = nltk.PorterStemmer()

def stemming(tokenized_text):
    text = [ps.stem(word) for word in tokenized_text]
    return text

movies_df['combined_features_stemmed'] = movies_df['combined_features_nostop'].apply(lambda x: stemming(x))

# movies_df.head(5)

In [0]:
len(movies_df.iloc[0,len(movies_df.columns) - 1])
# print(movies_df.iloc[12])

print("combined_features") 
print(movies_df.iloc[0, 10])

print("\ncombined_features_nostop") #Punctuation
print(movies_df.iloc[0, 11])

print("\ncombined_features_stemmed") #Steemed
print(movies_df.iloc[0, 12])

# movies_df.iloc['combined_features_nostop']
# movies_df.iloc['combined_features']


In [0]:
# Revisar tipos en una columna específica
tipo_por_fila = movies_df["combined_features_stemmed"].map(type)
print(tipo_por_fila.value_counts())

# LIMPIAR LISTAS EN STEEMED PARA PODER VECTORIZAR 

# 3. VECTORIZATION TF-IDF

In [0]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = movies_df["combined_features_stemmed"].fillna("").astype(str)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

In [0]:
# # 2. Start Vectorizator TF-IDF with N-gram = 2 (e.g. "science fiction")
#Convierte cada lista de tokens a un string antes de vectorizar:

# Convierte cada lista de tokens a un string antes de llamar a fit_transform.
corpus = movies_df["combined_features_stemmed"] \
    .fillna("") \
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=0.01,
    max_df=0.8
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print(f"Matriz generada. Forma: {tfidf_matrix.shape}")
type(tfidf_matrix)

In [0]:
DATA_DIR1 = Path("../models")
# # 4. Guardar los artefactos en Databricks para su uso posterior en el chatbot
# Es crucial guardar tanto la matriz como el vectorizador para procesar las consultas del usuario [3, 4]
joblib.dump(tfidf_vectorizer, DATA_DIR1 / 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, DATA_DIR1 / 'tfidf_matrix.pkl')

print("Artefactos guardados en la carpeta /models/")
print("")

In [0]:
### I tried to aply Cosine Similarity to check for similarities vectorially talking. The problem is at this point analize all of those movies and #it takes a lot of time and resources, so there is two ways to solve this 
#1. Calculate on-the-fly this will be good for the chatbot 
#2. Reduce the number of movies to analize. In this case could be the best ranked movies or the most popular ones

In [0]:
from sklearn.metrics.pairwise import linear_kernel

def recommend_on_the_fly(query_text, movies_df, vectorizer, tfidf_matrix):
    # 1. Transformar la consulta del usuario usando el mismo vectorizador
    query_vector = vectorizer.transform([query_text])
    
    # 2. Calcular similitud SOLO para esta consulta contra todas las películas
    # linear_kernel es más rápido que cosine_similarity para TF-IDF
    cosine_sim_query = linear_kernel(query_vector, tfidf_matrix).flatten()
    
    # 3. Obtener los índices de los 5 mejores resultados
    # argsort da los índices de menor a mayor, por eso usamos [-6:-1]
    related_indices = cosine_sim_query.argsort()[-6:-1][::-1]
    
    # 4. Retornar resultados
    return movies_df.iloc[related_indices][['title', 'genres_list', 'vote_average']]

In [0]:
%pip install streamlit
# Luego ejecuta: streamlit run app.py

In [0]:
# Celda 1: Instalar dependencias
%pip install -r /Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/requirements.txt

# Celda 2: Reiniciar Python (si es necesario)
dbutils.library.restartPython()

# Celda 3: Probar que todo se importa correctamente
import streamlit as st
import pandas as pd
import joblib
print("✓ Todas las dependencias instaladas correctamente")

In [0]:
# Agregar la ruta del proyecto al sys.path
import sys
sys.path.append('/Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/src')

# Importar el módulo del chatbot
from chatbot.chatbot_flow import chatbot_response, initialize_conversation_state  # type: ignore

print("✓ Módulo del chatbot importado correctamente")

In [0]:
import pandas as pd
import joblib

# Cargar el DataFrame de películas desde Unity Catalog
print("Cargando datos...")
movies_df = spark.read.table("workspace.datasets.movies_final").toPandas()

# Cargar el vectorizador TF-IDF
print("Cargando modelos...")
vectorizer = joblib.load('/Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/models/tfidf_vectorizer.pkl')

# Cargar o generar la matriz TF-IDF
try:
    tfidf_matrix = joblib.load('/Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/models/tfidf_matrix.pkl')
    print("✓ Matriz TF-IDF cargada desde archivo")
except:
    print("Generando matriz TF-IDF (esto puede tomar un momento)...")
    # Reconstruir el corpus
    corpus = movies_df["combined_features"] if "combined_features" in movies_df.columns else movies_df["title"]
    tfidf_matrix = vectorizer.transform(corpus)
    print("✓ Matriz TF-IDF generada")

print(f"\n✓ Todo listo: {len(movies_df)} películas cargadas")
print(f"  Matriz TF-IDF: {tfidf_matrix.shape}")

In [0]:
# Inicializar el estado de la conversación
state = initialize_conversation_state()

# Ejemplo 1: Usuario con preferencias claras
user_message = "Quiero ver algo de acción de los 90s"
print(f"Usuario: {user_message}")
print("\nProcesando...\n")

response, state = chatbot_response(user_message, state, movies_df, vectorizer, tfidf_matrix)

print(f"Bot: {response}")
print(f"\n--- Estado actualizado ---")
print(f"Géneros: {state.get('genres')}")
print(f"Año: {state.get('year')}")
print(f"Mood: {state.get('mood')}")
print(f"Rating: {state.get('rating')}")

In [0]:
# Ejemplo 2: Usuario sin suficiente información (el bot debe preguntar)
state2 = initialize_conversation_state()

user_message2 = "Hola, quiero ver una película"
print(f"Usuario: {user_message2}")
print("\nProcesando...\n")

response2, state2 = chatbot_response(user_message2, state2, movies_df, vectorizer, tfidf_matrix)

print(f"Bot: {response2}")
print(f"\n--- Estado actualizado ---")
print(f"Géneros: {state2.get('genres')}")
print(f"Año: {state2.get('year')}")

In [0]:
# Ejemplo 3: Usuario con preferencias complejas (mood + rating + década)
state3 = initialize_conversation_state()

user_message3 = "Quiero algo de comedia ligera y divertida de los 2000s con rating superior a 7"
print(f"Usuario: {user_message3}")
print("\nProcesando...\n")

response3, state3 = chatbot_response(user_message3, state3, movies_df, vectorizer, tfidf_matrix)

print(f"Bot: {response3}")
print(f"\n--- Estado actualizado ---")
print(f"Géneros: {state3.get('genres')}")
print(f"Año: {state3.get('year')}")
print(f"Mood: {state3.get('mood')}")
print(f"Rating: {state3.get('rating')}")

## 🚀 Lanzar la Aplicación Streamlit

Si los ejemplos anteriores funcionaron correctamente, ahora puedes desplegar la aplicación completa:

### Opción 1: Crear una Databricks App (Recomendado)
```bash
databricks apps create movie-recommender \
  --source-code-path /Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks
```

### Opción 2: Ejecutar localmente en el notebook
```python
# Nota: Streamlit en notebooks tiene limitaciones
!streamlit run /Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/app.py
```

In [0]:
!streamlit run /Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/app.py